# Desafío RNN para predecir una señal · Tablero docente

Usá este Colab para consultar los intentos enviados desde `RNN_signal_ejercicio.ipynb`. El ranking se ordena por menor MSE de validación y el scatter lo compara con la cantidad de parámetros entrenables. Usá el filtro de épocas para comparar corridas con el mismo presupuesto. El conjunto de test queda reservado para la evaluación final.

## 1. Preparar Google Sheets y Apps Script

1. Creá una hoja de cálculo con una pestaña vacía llamada `runs_rnn`. Copiá el ID de la hoja de cálculo que aparece entre `/d/` y `/edit` en su URL.
2. En **Extensiones > Apps Script**, pegá el código de la celda siguiente en `Code.gs`.
3. En **Configuración del proyecto > Propiedades de la secuencia de comandos**, agregá `SHEET_ID`, `COHORTE_ACTIVA` (por ejemplo `2026_2`) y `TOKEN_CLASE` (un valor largo y aleatorio).
4. Desplegá como **aplicación web**, con ejecución como propietario y acceso **Cualquier persona**. Compartí con los alumnos la URL que termina en `/exec`, la cohorte y el token.
5. Conservá la hoja privada. Iniciá sesión en este Colab con una cuenta que tenga acceso a ella.

El token habilita los envíos de la cohorte activa. Cada alumno informa su propio alias y el notebook envía las métricas calculadas en su sesión.

### Código para `Code.gs`

```javascript
const ENCABEZADOS = [
  'cohorte', 'alumno', 'run_id', 'timestamp', 'protocolo', 'modelo',
  'arquitectura', 'usar_derivada', 'n_parametros', 'epochs', 'train_mse',
  'val_mse', 'best_val_mse', 'final_val_mse', 'training_time',
];

function responder(datos) {
  return ContentService.createTextOutput(JSON.stringify(datos))
    .setMimeType(ContentService.MimeType.JSON);
}

function doPost(e) {
  const lock = LockService.getScriptLock();
  try {
    const p = JSON.parse(e.postData.contents);
    const props = PropertiesService.getScriptProperties();
    const token = props.getProperty('TOKEN_CLASE');
    const cohorte = props.getProperty('COHORTE_ACTIVA');
    const sheetId = props.getProperty('SHEET_ID');
    if (!token || !cohorte || !sheetId) throw Error('Faltan propiedades del servidor');
    if (!p || p.token !== token) throw Error('Token inválido');
    if (p.cohorte !== cohorte) throw Error('Cohorte inválida');
    const identificador = /^[\p{L}\p{N}][\p{L}\p{N} _.-]{0,59}$/u;
    if (!identificador.test(cohorte)) throw Error('Nombre de cohorte inválido');
    if (typeof p.alumno !== 'string' || !identificador.test(p.alumno)) {
      throw Error('Usá un alias de hasta 60 caracteres con letras, números, espacios, puntos o guiones');
    }
    if (p.protocolo !== 'RNN_SIGNAL_v1') throw Error('Protocolo inválido');
    if (typeof p.run_id !== 'string' || !/^[0-9a-f]{8}-(?:[0-9a-f]{4}-){3}[0-9a-f]{12}$/.test(p.run_id)) {
      throw Error('Identificador de corrida inválido');
    }
    if (typeof p.modelo !== 'string' || !/^[\p{L}\p{N}]/u.test(p.modelo) ||
        p.modelo.length > 80 || /[\r\n]/.test(p.modelo)) {
      throw Error('Usá un nombre de modelo de hasta 80 caracteres que empiece con una letra o un número');
    }
    if (typeof p.arquitectura !== 'string' || !p.arquitectura.trim() ||
        typeof p.usar_derivada !== 'boolean') throw Error('Arquitectura inválida');
    if (!Number.isSafeInteger(p.n_parametros) || p.n_parametros < 1 ||
        !Number.isSafeInteger(p.epochs) || p.epochs < 1 ||
        !Number.isFinite(p.training_time) || p.training_time <= 0) {
      throw Error('Cantidad de parámetros, épocas o tiempo inválido');
    }
    for (const clave of ['train_mse', 'val_mse']) {
      const historial = p[clave];
      if (!Array.isArray(historial) || historial.length !== p.epochs ||
          !historial.every(x => Number.isFinite(x) && x >= 0)) {
        throw Error('Historial inválido: ' + clave);
      }
    }

    const fila = {
      ...p,
      timestamp: new Date().toISOString(),
      arquitectura: JSON.stringify(p.arquitectura),
      usar_derivada: p.usar_derivada ? 'Sí' : 'No',
      train_mse: JSON.stringify(p.train_mse),
      val_mse: JSON.stringify(p.val_mse),
      best_val_mse: p.val_mse.reduce((menor, mse) => Math.min(menor, mse), Infinity),
      final_val_mse: p.val_mse[p.epochs - 1],
    };
    const valores = ENCABEZADOS.map(clave => fila[clave]);
    if (valores.some(valor => typeof valor === 'string' && valor.length > 50000)) {
      throw Error('El resultado supera el tamaño que admite una celda de Sheets');
    }

    if (!lock.tryLock(10000)) throw Error('Servidor ocupado; reintentá el envío');
    const hoja = SpreadsheetApp.openById(sheetId).getSheetByName('runs_rnn');
    if (!hoja) throw Error('Falta la pestaña runs_rnn');

    const ultimaFila = hoja.getLastRow();
    if (ultimaFila === 0) {
      hoja.appendRow(ENCABEZADOS);
    } else {
      const encabezado = hoja.getRange(1, 1, 1, ENCABEZADOS.length).getValues()[0];
      if (!ENCABEZADOS.every((clave, i) => clave === encabezado[i])) {
        throw Error('Los encabezados de runs_rnn no coinciden con el protocolo');
      }
    }
    if (ultimaFila > 1) {
      const encontrada = hoja.getRange(2, 3, ultimaFila - 1, 1)
        .createTextFinder(p.run_id).matchEntireCell(true).findNext();
      if (encontrada) return responder({ok: true, run_id: p.run_id, duplicado: true});
    }
    hoja.appendRow(valores);
    SpreadsheetApp.flush();
    return responder({ok: true, run_id: p.run_id, duplicado: false});
  } catch (err) {
    return responder({ok: false, error: String(err.message)});
  } finally {
    if (lock.hasLock()) lock.releaseLock();
  }
}
```

## 2. Conectar este Colab

Completá `SHEET_ID` y ejecutá las celdas. El tablero lee con los permisos de tu cuenta de Google.

In [ ]:
%pip -q install gspread ipywidgets

In [ ]:
import json
import math

import google.auth
import gspread
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import auth, output
from IPython.display import clear_output, display
from matplotlib.colors import to_hex

output.enable_custom_widget_manager()

SHEET_ID = "1D_eO8jfN-zTHjWXTLHddSXPu9hudKZWx8fEJzL2QSkU"
hoja = None
if SHEET_ID.strip():
    auth.authenticate_user()
    credenciales, _ = google.auth.default()
    hoja = gspread.authorize(credenciales).open_by_key(SHEET_ID.strip()).worksheet("runs_rnn")
else:
    print("Completá SHEET_ID para conectar el tablero.")

## 3. Leer y ordenar intentos

In [ ]:
PROTOCOLO = "RNN_SIGNAL_v1"
ENCABEZADOS = (
    "cohorte", "alumno", "run_id", "timestamp", "protocolo", "modelo",
    "arquitectura", "usar_derivada", "n_parametros", "epochs", "train_mse",
    "val_mse", "best_val_mse", "final_val_mse", "training_time",
)


def cargar_resultados():
    if hoja is None:
        raise ValueError("Completá SHEET_ID y ejecutá la celda de conexión.")
    filas = hoja.get_all_values(value_render_option="UNFORMATTED_VALUE")
    if not filas:
        return pd.DataFrame(), []
    if tuple(filas[0][:len(ENCABEZADOS)]) != ENCABEZADOS:
        raise ValueError("Los encabezados de runs_rnn no coinciden con el protocolo.")

    registros, errores = [], []
    vistos = set()
    for numero, valores in enumerate(filas[1:], start=2):
        if not any(valores):
            continue
        fila = dict(zip(ENCABEZADOS, valores))
        try:
            if fila["protocolo"] != PROTOCOLO:
                continue
            for clave in ("n_parametros", "epochs"):
                entero = float(fila[clave])
                if not entero.is_integer() or entero < 1:
                    raise ValueError(f"{clave} debe ser un entero positivo")
                fila[clave] = int(entero)
            for clave in ("cohorte", "alumno", "run_id", "timestamp", "modelo"):
                fila[clave] = str(fila[clave]).strip()
                if not fila[clave]:
                    raise ValueError(f"Falta {clave}")
            if fila["run_id"] in vistos:
                raise ValueError("run_id duplicado")
            fila["arquitectura"] = json.loads(fila["arquitectura"])
            if not isinstance(fila["arquitectura"], str) or fila["usar_derivada"] not in ("Sí", "No"):
                raise ValueError("Arquitectura o derivada inválida")
            for clave in ("train_mse", "val_mse"):
                historia = json.loads(fila[clave])
                if (not isinstance(historia, list) or len(historia) != fila["epochs"] or
                    not all(type(x) in (int, float) and math.isfinite(x) and x >= 0 for x in historia)):
                    raise ValueError(f"Historial inválido: {clave}")
                fila[clave] = historia
            fila["best_val_mse"] = min(fila["val_mse"])
            fila["final_val_mse"] = fila["val_mse"][-1]
            fila["training_time"] = float(fila["training_time"])
            if not math.isfinite(fila["training_time"]) or fila["training_time"] <= 0:
                raise ValueError("Tiempo inválido")
            registros.append(fila)
            vistos.add(fila["run_id"])
        except (KeyError, TypeError, ValueError, OverflowError) as exc:
            errores.append(f"Fila {numero}: {exc}")
    return pd.DataFrame(registros), errores


def ordenar_resultados(df, mejor_por_alumno=False):
    df = df.sort_values(["best_val_mse", "timestamp", "run_id"])
    if mejor_por_alumno:
        df = df.drop_duplicates(["cohorte", "alumno"], keep="first")
    return df

## 4. Tablero

Pulsá **Actualizar resultados** para volver a leer la hoja. Filtrá por cohorte, épocas y todos los intentos o el mejor por alumno. El selector permite mostrar las curvas de hasta cinco corridas y abrir sus detalles de arquitectura.

El borde del ranking, el punto y las curvas de cada intento usan el mismo color. Las curvas discontinuas muestran entrenamiento y las continuas, validación. Los puntos grises corresponden a los demás intentos fuera de las primeras 20 filas y de la selección.

In [ ]:
def dibujar_resultados(df, seleccion):
    visibles = df.head(20)
    elegidas = df[df["run_id"].isin(seleccion[:5])]
    ids_coloreados = pd.concat([visibles["run_id"], elegidas["run_id"]]).unique()
    tab20 = plt.get_cmap("tab20").colors
    paleta = tab20[::2] + tab20[1::2] + plt.get_cmap("Dark2").colors
    colores = {run_id: to_hex(color) for run_id, color in zip(ids_coloreados, paleta)}

    columnas = {
        "cohorte": "Cohorte", "alumno": "Alumno", "modelo": "Modelo", "run_id": "Intento",
        "usar_derivada": "Derivada", "n_parametros": "Parámetros", "epochs": "Épocas",
        "best_val_mse": "Mejor MSE val.", "final_val_mse": "MSE val. final", "training_time": "Tiempo (s)",
    }
    ranking = visibles[list(columnas)].rename(columns=columnas)
    ranking.insert(0, "Puesto", range(1, len(ranking) + 1))
    bordes = visibles["run_id"].map(lambda run_id: f"border-left: 5px solid {colores[run_id]}")
    display(ranking.style.apply(lambda _: bordes, subset="Puesto").format({
        "Intento": lambda run_id: run_id[:8], "Mejor MSE val.": "{:.6f}",
        "MSE val. final": "{:.6f}", "Tiempo (s)": "{:.1f}",
    }, escape="html").hide(axis="index"))

    fig, (curvas, scatter) = plt.subplots(1, 2, figsize=(14, 4.7), layout="constrained")
    for r in elegidas.itertuples():
        epocas = range(1, r.epochs + 1)
        color = colores[r.run_id]
        curvas.plot(epocas, r.train_mse, "--", color=color)
        curvas.plot(epocas, r.val_mse, color=color, label=f"{r.alumno} · {r.run_id[:8]}")
    curvas.set(title="MSE por época", xlabel="Época", ylabel="MSE")
    if len(elegidas):
        curvas.legend(fontsize=8)

    color_punto = df["run_id"].map(colores)
    grises = df[color_punto.isna()]
    destacados = df[color_punto.notna()]
    if not grises.empty:
        scatter.scatter(grises["n_parametros"], grises["best_val_mse"], color="#b0b0b0")
    scatter.scatter(destacados["n_parametros"], destacados["best_val_mse"],
                    c=color_punto.dropna(), edgecolors="white", linewidths=0.5)
    scatter.set(title="MSE de validación vs. parámetros",
                xlabel="Parámetros entrenables", ylabel="Mejor MSE de validación")
    for ax in (curvas, scatter):
        ax.grid(alpha=0.25)
        ax.set_axisbelow(True)
    plt.show()
    plt.close(fig)

    if len(elegidas):
        detalles = widgets.Accordion(children=[
            widgets.Textarea(value=r.arquitectura, disabled=True,
                             layout=widgets.Layout(width="100%", height="130px"))
            for r in elegidas.itertuples()
        ], selected_index=None)
        for i, r in enumerate(elegidas.itertuples()):
            detalles.set_title(i, f"{r.alumno} · {r.modelo} · {r.run_id[:8]}")
        display(detalles)

In [ ]:
boton = widgets.Button(description="Actualizar resultados", button_style="primary")
cohorte = widgets.Dropdown(description="Cohorte", options=[("Todas", None)])
epocas = widgets.Dropdown(description="Épocas", options=[("Todas", None)])
modo = widgets.ToggleButtons(description="Intentos", options=[("Todos", False), ("Mejor por alumno", True)])
selector = widgets.SelectMultiple(description="Curvas", rows=7, layout=widgets.Layout(width="95%"))
estado = widgets.Label()
salida = widgets.Output()
datos = pd.DataFrame()
advertencias = []
actualizando = False


def vista_filtrada():
    df = datos
    if df.empty:
        return df
    if cohorte.value is not None:
        df = df[df["cohorte"] == cohorte.value]
    if epocas.value is not None:
        df = df[df["epochs"] == epocas.value]
    return ordenar_resultados(df, mejor_por_alumno=modo.value)


def redibujar(_=None):
    if actualizando:
        return
    with salida:
        clear_output(wait=True)
        df = vista_filtrada()
        if df.empty:
            print("No hay resultados para los filtros seleccionados.")
        else:
            if len(selector.value) > 5:
                print("Se muestran las primeras cinco corridas seleccionadas.")
            dibujar_resultados(df, selector.value)
        if advertencias:
            print(f"Filas omitidas ({len(advertencias)}):")
            for aviso in advertencias[:10]:
                print(aviso)


def actualizar_controles(recalcular_filtros=False):
    global actualizando
    actualizando = True
    try:
        if recalcular_filtros:
            for control, columna in ((cohorte, "cohorte"), (epocas, "epochs")):
                anterior = control.value
                valores = sorted(datos[columna].unique().tolist()) if not datos.empty else []
                control.options = [("Todas", None)] + [(str(v), v) for v in valores]
                control.value = anterior if anterior in valores else None

        seleccion_anterior = selector.value
        opciones = [
            (f"{r.cohorte} · {r.alumno} · {r.modelo} · {r.best_val_mse:.6f} · {r.run_id[:8]}", r.run_id)
            for r in vista_filtrada().itertuples()
        ]
        ids = {valor for _, valor in opciones}
        conservadas = tuple(valor for valor in seleccion_anterior if valor in ids)
        selector.options = opciones
        selector.value = conservadas or tuple(valor for _, valor in opciones[:5])
    finally:
        actualizando = False
    redibujar()


def cambiar_filtro(_):
    if not actualizando:
        actualizar_controles()


def refrescar(_=None):
    global datos, advertencias
    try:
        datos, advertencias = cargar_resultados()
    except Exception as exc:
        estado.value = f"No se pudo leer Google Sheets: {exc}"
        return
    estado.value = f"{len(datos)} intentos cargados"
    actualizar_controles(recalcular_filtros=True)


boton.on_click(refrescar)
for control in (cohorte, epocas, modo):
    control.observe(cambiar_filtro, names="value")
selector.observe(redibujar, names="value")
controles = widgets.HBox([boton, cohorte, epocas, modo], layout=widgets.Layout(flex_flow="row wrap"))
display(widgets.VBox([controles, selector, estado, salida]))
refrescar()